# E · Phân tích fidelity, coverage và validity độc lập

**Câu hỏi:** đường có là lý do hợp lý cho truy vấn–ảnh không? Notebook CPU đọc một lượt `path-*` hoàn tất. Bootstrap theo **query**, giữ seed trong cụm; coverage tính trên mọi case đăng ký, kể cả `no_relational_explanation`. Fidelity chỉ so với random path khớp cấu trúc khi có đối chứng độc lập. Logic và căn cứ ảnh chờ hai người chấm; score drop không tự cho nhãn validity.

In [ ]:
from pathlib import Path
import json,sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive/graphml-project3').is_dir() else next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'graphml-project3').is_dir())
DUONG=ROOT/'graphml-project3/duong'; RUNS=DUONG/'runs/qa_vg_psg_xai'
sys.path.insert(0,str(DUONG))
from xai.review_workbook import create_review_workbook
from xai.image_download import fetch_review_image
RESULTS_DIR=None  # set an explicit completed path-* directory for the report
if RESULTS_DIR is None:
    complete=sorted(p.parent for p in RUNS.glob('path-*/complete.json'))
    assert complete,'Run notebook 02 first.'
    RESULTS_DIR=complete[-1]
RESULTS_DIR=Path(RESULTS_DIR)
results=pd.read_csv(RESULTS_DIR/'xai_results.csv')
cases=pd.read_csv(RESULTS_DIR/'case_manifest.csv')
paths=[json.loads(line) for line in (RESULTS_DIR/'paths.jsonl').open(encoding='utf8')]
meta=json.loads((RESULTS_DIR/'complete.json').read_text())
assert len(results)==meta['n_result_rows'] and len(paths)==meta['n_path_rows']
analysis=RESULTS_DIR/'analysis'; analysis.mkdir(exist_ok=True)
print(RESULTS_DIR,len(results),len(cases),len(paths))


## 1 · Denominator, path yield và random matching

Mỗi `dataset × screen limit × budget` giữ cùng số case. Sparsity = 1 − số cạnh explanation duy nhất / `E_eligible`; `E_screened / E_eligible` được báo riêng. Nếu `E_eligible_relation = 0`, relation fraction là N/A. Random thiếu cấu trúc khớp được báo thiếu đối chứng.

In [ ]:
main=results[results.intervention=='relational_path'].copy()
assert not main.duplicated(['dataset','query_id','model_seed','screen_path_limit','budget']).any()
assert (main.n_screened<=main.n_eligible).all() and (main.n_removed_unique<=main.budget).all()
main['matched_advantage']=main.final_score_drop-main.random_final_drop_mean
coverage=main.groupby(['dataset','screen_path_limit','budget']).agg(n_cases=('query_id','size'),n_queries=('query_id','nunique'),positive_path_rate=('has_positive_relational_path','mean'),selected_path_rate=('n_paths',lambda x:(x>0).mean()),no_explanation_rate=('status',lambda x:(x!='ok').mean()),structural_no_path_rate=('status',lambda x:(x=='no_relational_path').mean()),mean_eligible=('n_eligible','mean'),mean_screen_coverage=('screen_coverage','mean'),mean_sparsity=('sparsity','mean'),matched_cases=('random_n',lambda x:(x>0).sum()),unique_random_sets=('random_unique_n','sum'),mean_fidelity_advantage=('matched_advantage','mean')).reset_index()
display(coverage)
display(main.groupby(['dataset','screen_path_limit','budget','group_source']).status.value_counts())


## 2 · CI 95% theo query và hình fidelity–sparsity–coverage

Lấy lại query ID, không lấy lại từng dòng seed. CI fidelity chỉ tính trên query có random match và đi kèm `n_matched_queries`. Yield/coverage dùng toàn bộ query. Không so trực tiếp thang fuse VG và PSG.

In [ ]:
rng=np.random.default_rng(20261121); ci=[]
for key,part in main.groupby(['dataset','screen_path_limit','budget']):
    by_query=part.groupby('query_id').agg(coverage=('n_paths',lambda x:(x>0).mean()),advantage=('matched_advantage','mean'))
    out={'dataset':key[0],'screen_path_limit':key[1],'budget':key[2],'n_queries':len(by_query),'n_matched_queries':by_query.advantage.notna().sum()}
    for field in ('coverage','advantage'):
        values=by_query[field].dropna().to_numpy(); out[field+'_mean']=float(values.mean()) if len(values) else np.nan
        if len(values):
            draws=rng.choice(values,size=(5000,len(values)),replace=True).mean(axis=1)
            out[field+'_ci_low'],out[field+'_ci_high']=map(float,np.quantile(draws,[.025,.975]))
        else: out[field+'_ci_low']=out[field+'_ci_high']=np.nan
    ci.append(out)
ci=pd.DataFrame(ci); display(ci)
coverage.to_csv(analysis/'coverage.csv',index=False); ci.to_csv(analysis/'cluster_bootstrap_ci.csv',index=False)
fig,axes=plt.subplots(1,3,figsize=(15,4))
for (dataset,limit),sub in coverage.groupby(['dataset','screen_path_limit']):
    sub=sub.sort_values('budget'); label=f'{dataset.upper()} screen={limit}'
    for ax,y in zip(axes,['mean_fidelity_advantage','mean_sparsity','selected_path_rate']): ax.plot(sub.budget,sub[y],marker='o',label=label)
for ax,title in zip(axes,['Matched final drop advantage','Sparsity / eligible edges','Relational yield / all cases']):
    ax.set_title(title); ax.set_xlabel('Unique-edge budget'); ax.grid(alpha=.3)
axes[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(analysis/'fidelity_sparsity_coverage.png',dpi=160)
display(fig); plt.close(fig)


In [ ]:
report=f"""# Path XAI report — {RESULTS_DIR.name}

Protocol: {json.loads((RESULTS_DIR/'protocol.json').read_text())['query_mode']}; fixed top-100 candidates, frozen QA checkpoints and alpha.
The current scorer consumes edge_index, not predicate identity; relational paths describe connectivity only.

## Coverage and fidelity by dataset, screen and unique-edge budget

```
{coverage.to_string(index=False)}
```

Matched fidelity is conditional on an independent random path with the same edge-type/length signature and unique-edge budget. The `matched_cases` column gives its denominator. `mean_sparsity` is conditional on an explanation existing; no-path cases remain in yield/coverage denominators. VG and PSG use different graph/model configurations, so their fuse-score magnitudes are not directly compared.

## Query-cluster bootstrap 95% CI

```
{ci.to_string(index=False)}
```

Logical and visual validity remain pending independent review. Structural validity confirms graph continuity only. `no_relational_explanation` is a result, not an exclusion.
"""
(analysis/'report.md').write_text(report,encoding='utf8')
print('Report:',analysis/'report.md')


## 3 · 10 case study và Excel chấm ảnh

Chọn 5 success/5 failure theo seed 0 và screen lớn nhất, gồm cả case không có đường. Tải GT và top-1 từ URL nguồn, nhúng vào workbook khi tải được; giữ URL nếu mạng lỗi. Các cột đánh giá và rubric để trống. Hai người chấm riêng từng bước: đối tượng, predicate/chiều, thông tin mới, căn cứ ảnh, toàn path. Mỗi bản là một reviewer độc lập.

In [ ]:
limit=int(cases.screen_path_limit.max()); pool=cases[(cases.model_seed==0)&(cases.screen_path_limit==limit)]
def choose_outcome(outcome):
    part=pool[pool.retrieval_outcome==outcome].sort_values(['dataset','query_id'])
    preferred=pd.concat([x.head(2) for _,x in part.groupby('dataset')]) if len(part) else part
    remaining=part.drop(preferred.index).sort_values(['query_id','dataset'])
    return pd.concat([preferred,remaining.head(max(0,5-len(preferred)))]).head(5)
chosen=pd.concat([choose_outcome('success'),choose_outcome('failure')])
assert len(chosen)==10,'Need five success and five failure.'
image_dir=analysis/'case_images'; image_dir.mkdir(exist_ok=True)
ALLOW_UNVERIFIED_TLS=True  # public dataset images only; provenance is saved below
path_index={}
for p in paths:
    if p.get('model_seed')==0 and p.get('screen_path_limit')==limit: path_index.setdefault((p['dataset'],str(p['query_id'])),[]).append(p)
review_cases=[]; review_steps=[]; download_records=[]
for row in chosen.to_dict('records'):
    ps=[p for p in path_index.get((row['dataset'],str(row['query_id'])),[]) if 3 in p.get('selected_at_budgets',[])]
    row['explanation_level']='relational_connectivity' if ps else 'no_relational_explanation'
    row['path_or_edge_text']='; '.join(' → '.join(p['node_ids'])+' ['+','.join(p['predicates'])+']' for p in ps) or 'no path selected'
    for role,url_key,file_key in [('gt','gt_source_url','gt_image_file'),('retrieved','retrieved_source_url','retrieved_image_file')]:
        path=image_dir/f"{row['dataset']}_{row['query_id']}_{role}.jpg"
        try:
            info=fetch_review_image(row[url_key],path,allow_unverified_tls=ALLOW_UNVERIFIED_TLS)
            row[file_key]=str(path)
            download_records.append({'dataset':row['dataset'],'query_id':row['query_id'],'role':role,**info})
        except Exception as exc:
            row[file_key]=''; download_records.append({'dataset':row['dataset'],'query_id':row['query_id'],'role':role,'source_url':row[url_key],'status':'failed','error':str(exc)})
            print('Image unavailable:',row['query_id'],role,str(exc)[:180])
    review_cases.append(row)
    if not ps: review_steps.append({'dataset':row['dataset'],'query_id':row['query_id'],'target_image_id':row['retrieved_image_id'],'path_id':'no_path'})
    for p in ps:
        for i,e in enumerate(p['edge_ids']):
            review_steps.append({'dataset':row['dataset'],'query_id':row['query_id'],'target_image_id':row['retrieved_image_id'],'path_id':p['path_id'],'step_index':i+1,'edge_id':e,'source_node':p['node_ids'][i],'target_node':p['node_ids'][i+1],'predicate_hint':p['predicates'][i],'edge_type':p['edge_types'][i],'edge_provenance':p['sources'][i],'model_score_drop':p['final_score_drop']})
workbook=create_review_workbook(review_cases,review_steps,analysis/'validity_case_review_10.xlsx',analysis)
pd.DataFrame(download_records).to_csv(analysis/'image_download_manifest.csv',index=False)
print('Downloaded/available images:',sum(r['status']!='failed' for r in download_records),'/',len(download_records))
annotations=pd.DataFrame(review_steps)
for col in ['reviewer','object_supported','predicate_direction_valid','new_information','visual_evidence','step_decision','support_source','evidence_reason','whole_path_decision','adjudicated_decision']:
    annotations[col]=''
annotations.to_csv(analysis/'validity_annotations.csv',index=False)
pd.DataFrame(review_cases).to_csv(analysis/'case_studies_10.csv',index=False)
print('Review workbook:',workbook,'step rows:',len(review_steps))


## 4 · Hai reviewer và hòa giải

Sau khi chấm độc lập, chỉ định hai workbook. Tính tỷ lệ đồng thuận và Cohen κ trên `step_decision` gồm cả `uncertain`. Đưa bất đồng vào bảng hòa giải; chỉ báo logical/visual supported sau khi `adjudicated_decision` đầy đủ. Không dùng graph schema hoặc provenance làm nhãn ảnh.

In [ ]:
REVIEWER_A=None  # Path('.../reviewer_a.xlsx')
REVIEWER_B=None  # Path('.../reviewer_b.xlsx')
if REVIEWER_A and REVIEWER_B:
    keys=['dataset','query_id','target_image_id','path_id','step_index','edge_id']
    a=pd.read_excel(REVIEWER_A,sheet_name='Step review',keep_default_na=False)
    b=pd.read_excel(REVIEWER_B,sheet_name='Step review',keep_default_na=False)
    ab=a.merge(b,on=keys,suffixes=('_a','_b'),validate='one_to_one')
    assert len(ab)==len(a)==len(b)
    assert (ab.step_decision_a!='').all() and (ab.step_decision_b!='').all()
    labels=['supported','contradicted','uncertain']
    assert ab.step_decision_a.isin(labels).all() and ab.step_decision_b.isin(labels).all()
    po=(ab.step_decision_a==ab.step_decision_b).mean()
    pe=sum((ab.step_decision_a==label).mean()*(ab.step_decision_b==label).mean() for label in labels)
    kappa=(po-pe)/(1-pe) if pe<1 else float('nan')
    print('Agreement:',po,'Cohen kappa:',kappa,'n:',len(ab))
    ab.to_csv(analysis/'validity_annotations_two_reviewers.csv',index=False)
else:
    print('Validity pending independent human annotation; no logical/visual support rate reported.')
ADJUDICATED_WORKBOOK=None  # Path('.../adjudicated_validity.xlsx')
if ADJUDICATED_WORKBOOK:
    final=pd.read_excel(ADJUDICATED_WORKBOOK,sheet_name='Step review',keep_default_na=False)
    assert len(final)==len(review_steps)
    assert final.adjudicated_decision.isin(['supported','contradicted','uncertain']).all()
    final.to_csv(analysis/'validity_annotations.csv',index=False)
    real=final[final.path_id!='no_path']
    for field in ['object_supported','predicate_direction_valid','new_information','visual_evidence','whole_path_decision']:
        assert real[field].isin(['supported','contradicted','uncertain']).all(),field
    real=real.copy()
    real['logical_supported']=real[['predicate_direction_valid','new_information','adjudicated_decision']].eq('supported').all(axis=1)
    real['visual_supported']=real[['object_supported','visual_evidence','adjudicated_decision']].eq('supported').all(axis=1)
    real['step_known']=real.adjudicated_decision.ne('uncertain')
    path_scores=real.groupby(['dataset','query_id','path_id']).agg(
        logical_steps=('logical_supported','all'),visual_steps=('visual_supported','all'),
        known_steps=('step_known','all'),whole_supported=('whole_path_decision',lambda x:x.eq('supported').all()),
        whole_known=('whole_path_decision',lambda x:x.ne('uncertain').all())).reset_index()
    path_scores['supported']=path_scores.logical_steps & path_scores.visual_steps & path_scores.whole_supported
    path_scores['known']=path_scores.known_steps & path_scores.whole_known
    path_scores.to_csv(analysis/'validity_path_scores.csv',index=False)
    print('Known step rate:',real.step_known.mean(),'Logical step support:',real.logical_supported.mean(),
          'Visual step support:',real.visual_supported.mean(),'Known path rate:',path_scores.known.mean(),
          'Supported path rate:',path_scores.supported.mean(),
          'No-path cases:',(final.path_id=='no_path').sum())
    with (analysis/'report.md').open('a',encoding='utf8') as f:
        f.write(f"""

## Independently adjudicated validity (10 case studies)

Step known: {real.step_known.mean():.3f}; logical step support: {real.logical_supported.mean():.3f}; visual step support: {real.visual_supported.mean():.3f}.
Path known: {path_scores.known.mean():.3f}; whole-path support: {path_scores.supported.mean():.3f}; no-path cases: {(final.path_id=='no_path').sum()}.
These rates use every selected step/path in the 10-case workbook; uncertain counts in the denominator.
""")
